# 01 — Data Overview & Cross-Market Microstructure Summary

This notebook provides a first look at the Destiny Research dataset.
It covers:
1. **Data inventory** — products available on disk, date ranges, contracts
2. **Product configuration** — exchange, currency, tick size, session hours
3. **Cross-market microstructure summary** — order-to-trade ratio, cancel rate, fill rate and traded price range across US, European, and Asian equity index futures

All data is loaded via the `DestinyResearch` package (`dr`), which provides a unified access layer over the normalized MBO and reconstructed MBP-1 parquet files.

> **Universe:** 10 products — 5 exchanges — 5 currencies — 3 geographic zones  
> **Data source:** Databento (ES, NIY, NKD, FDAX, FESX, FSMI) + HKEX OMD-D (HSI, MHI, HHI, MCH)  
> **Schema:** provider-agnostic MBO (Market-By-Order), reconstructed MBP-1 (top-of-book snapshots)


## Setup

In [1]:
import os
os.chdir('/home/julien/repo/destiny-research')

import DestinyResearch as dr
import pandas as pd

dr.overview()

  DestinyResearch — Data Overview
  ES      —   27 days  [2025-10-01 → 2025-10-31]
  FDAX    —   21 days  [2025-05-02 → 2025-05-30]
  FESX    —   21 days  [2025-05-02 → 2025-05-30]
  FSMI    —   21 days  [2025-05-02 → 2025-05-30]
  HHI     —   20 days  [2026-02-02 → 2026-02-27]
  HSI     —   20 days  [2026-02-02 → 2026-02-27]
  MCH     —   20 days  [2026-02-02 → 2026-02-27]
  MHI     —   20 days  [2026-02-02 → 2026-02-27]
  NIY     —  363 days  [2025-01-01 → 2026-02-27]
  NKD     —  363 days  [2025-01-01 → 2026-02-27]

Quick start:
  tbl = dr.get_mbo_front('ES', '2025-10-01')
  tbl = dr.get_mbp1('ESZ25', '2025-10-01', columns=['ts_recv','bid_px_00','ask_px_00'])
  df  = dr.to_df(tbl)
  dr.show_info('HSI')


The dataset spans three geographic zones with different data windows:
- **US (CME):** E-mini S&P 500 — October 2025 (1 month)
- **Europe (Eurex):** DAX, Euro Stoxx 50, SMI — May 2025 (1 month)
- **Asia (CME + HKEX):** Nikkei 225 JPY/USD — full year 2025 + January–February 2026; Hang Seng and H-shares futures — February 2026 (1 month)

NIY and NKD show 363 trading days because CME Globex runs ~22 hours/day, including Sunday evenings (US time) which correspond to Monday morning in Tokyo — yielding approximately 6 sessions per week rather than 5.

A common cross-market date will become available as more data is acquired. The current analysis uses the most representative single day per zone.


## Product Configuration

`dr.show_info()` returns the full market configuration for a product: exchange, currency, contract multiplier, tick size, session hours (UTC), and RTH definition. All parameters are sourced from `ingestion/market_config.py`.

In [2]:
for product in dr.get_available_products():
    dr.show_info(product)

────────────────────────────────────────────────────────────
  ES  —  E-mini S&P 500 Future (CME Globex)
────────────────────────────────────────────────────────────
  Exchange      : CME  |  Provider: databento
  Currency      : USD  |  Point value: 50.0 USD/pt
  Tick size     : 0.2500 pts
  Price floor   : 500 pts
  Timezone      : America/New_York
  Session UTC   : 22:00:00 → 21:00:00
  RTH EDT (UTC) : 13:30:00 → 20:00:00
  RTH EST (UTC) : 14:30:00 → 21:00:00
  Contracts     : ['ESH26', 'ESH27', 'ESH28', 'ESH29', 'ESH30', 'ESM26', 'ESM27', 'ESM28', 'ESM29', 'ESM30', 'ESU26', 'ESU27', 'ESU28', 'ESU29', 'ESU30', 'ESZ25', 'ESZ26', 'ESZ27', 'ESZ28', 'ESZ29', 'ESZ30', 'ES_CAL_H26H27', 'ES_CAL_H26M26', 'ES_CAL_H26U26', 'ES_CAL_H26Z26', 'ES_CAL_H27M27', 'ES_CAL_H27U27', 'ES_CAL_M26H27', 'ES_CAL_M26U26', 'ES_CAL_M26Z26', 'ES_CAL_M27U27', 'ES_CAL_U26H27', 'ES_CAL_U26M27', 'ES_CAL_U26Z26', 'ES_CAL_U27Z27', 'ES_CAL_Z25H26', 'ES_CAL_Z25M26', 'ES_CAL_Z25U26', 'ES_CAL_Z25Z26', 'ES_CAL_Z26H27', 'E

## DestinyResearch API — Quick Reference

The `dr` module provides a unified access layer with automatic path resolution, front-month detection, and memory-efficient PyArrow loading.

Key design principles:
- All `get_*` functions return `pyarrow.Table` — columnar, memory-efficient, zero-copy where possible
- Use `columns=` to restrict which columns are read from disk (Parquet column pruning)
- Use `dr.to_df()` explicitly when pandas operations are needed
- Front-month detection uses trade count via DuckDB (`method='volume'`, default) or expiry parsing (`method='expiry'`)


### Inspect schema without loading data

In [3]:
dr.schema('ESZ25', '2025-10-01', kind='mbp1')

ts_event: uint64 not null
ts_recv: uint64 not null
action: string not null
side: string not null
price: double not null
flags: uint8 not null
sequence: uint32 not null
bid_px_00: double
ask_px_00: double
bid_sz_00: uint32
ask_sz_00: uint32
bid_ct_00: uint32
ask_ct_00: uint32

The MBP-1 schema contains one snapshot per `F_LAST` event (i.e. every time the top-of-book changes):
- `bid_px_00` / `ask_px_00`: best bid/ask price (level 0), in index points
- `bid_sz_00` / `ask_sz_00`: quantity available at best bid/ask, in contracts
- `bid_ct_00` / `ask_ct_00`: number of orders resting at best bid/ask
- `action` / `side`: the event that triggered this snapshot (ADD/CANCEL/MODIFY/TRADE on BID/ASK)
- `ts_recv`: reception timestamp in nanoseconds UTC (primary sort key)


### Load MBP-1 data with column selection

In [4]:
tbl = dr.get_mbp1('ESZ25', '2025-10-01', columns=['ts_recv','bid_px_00','ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00', 'ask_ct_00'])

In [5]:
tbl.to_pandas().head()

,ts_recv,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00
0,1759276800000922600,6723.25,6723.5,16,8,12,7
1,1759276800000931585,6723.25,6723.5,16,8,12,7
2,1759276800000938658,6723.25,6723.5,16,8,12,7
3,1759276800007408987,6723.25,6723.5,17,8,13,7
4,1759276800026466653,6723.25,6723.5,17,6,13,6


The first few rows of ESZ25 on 2025-10-01 show the top-of-book at market open. The spread is 0.25 pts (one tick), with 16 contracts bid and 8 offered at the best level. `bid_ct_00=12` means 12 distinct orders are resting at the best bid — a relatively thin queue consistent with the opening period before liquidity builds.

Note that `ts_recv` is stored as nanoseconds since Unix epoch. Use `dr.with_datetime(tbl)` to add a human-readable UTC datetime column for plotting.


### Front-month detection

In [6]:
dr.get_front_contract("ES", "2025-10-01")

'ESZ25'

In [7]:
dr.get_front_contract("ES", "2025-10-01", method="expiry") # method: volume (default), expiry

'ESZ25'

Front-month detection uses two methods:
- **`volume` (default):** identifies the contract with the highest trade count on the given day, via a single DuckDB COUNT query on the normalized MBO files. This is the most robust definition — the front month is the contract where price discovery actually occurs.
- **`expiry`:** parses the contract ticker to find the nearest expiry after the given date. Faster (no disk read) but less reliable when multiple contracts trade actively near roll date.

Both methods agree on `ESZ25` for October 2025 — the December 2025 expiry was the clear front month throughout Q4 2025.


### Load MBO data (automatic front-month selection)

In [8]:
tbl = dr.get_mbo_front('FDAX', '2025-05-14')

[dr] Front month for FDAX on 2025-05-14: FDAXM25


In [9]:
tbl.shape

(2324424, 14)

FDAXM25 on 2025-05-14 contains **2.3M MBO events** across 14 columns. This is a representative day for the DAX future — the high order count relative to trades (OTR ~44:1, as we will see below) reflects Eurex's algorithmic market making activity.

The MBO schema preserves every order lifecycle event: ADD, CANCEL, MODIFY, TRADE, FILL, CLEAR. This granularity is what allows precise computation of order flow imbalance (OFI), VPIN, and other microstructure signals that require individual order tracking.


---

## Cross-Market Microstructure Summary

We compute key microstructure metrics for all 10 products across three geographic zones. Due to different data acquisition periods, each zone uses its own reference date:

| Zone | Products | Reference date | Notes |
|------|----------|---------------|-------|
| US   | ES | 2025-10-10 | October 2025 dataset |
| EU   | FDAX, FESX, FSMI | 2025-05-14 | May 2025 dataset |
| Asia | NIY, NKD, HSI, MHI, HHI, MCH | 2026-02-02 | First common date across all Asian products |

**Metrics computed (via DuckDB, single pass per file, no data loaded into RAM):**
- `n_trade`: number of executed trades
- `otr`: order-to-trade ratio = n_add / n_trade — measures market noisiness
- `cancel_rate`: n_cancel / n_add — fraction of submitted orders that are cancelled
- `fill_rate`: n_trade / n_add — fraction of submitted orders that result in a trade
- `price_min_pts` / `price_max_pts`: intraday traded price range in index points

**Note on `cancel_rate` and `fill_rate`:** these two metrics are independent and do not sum to 1. A single order can contribute to both (partially filled then cancelled). `cancel_rate > 1` on Eurex products reflects overnight GTC orders whose ADD event falls outside our daily window — these orphan CANCELs are preserved in LOOSE validation mode. OTR is the more robust cross-exchange comparison metric.


### United States — CME Globex

In [10]:
df_us = dr.get_cross_market_summary(["ES"], "2025-10-10")
df_us.insert(0, "zone", "US")
df_us

,zone,product,contract,date,exchange,currency,n_rows,n_add,n_cancel,n_modify,n_trade,otr,cancel_rate,fill_rate,price_min_pts,price_max_pts,tick_size_pts,point_value
0,US,ES,ESZ25,2025-10-10,CME,USD,19696159,7453354,7447317,2043397,885379,8.42,0.9992,0.1188,6540.25,6806.5,0.25,50.0


**ES (E-mini S&P 500) — 2025-10-10**

The E-mini S&P 500 is the most liquid equity index future in the world, and our metrics confirm its microstructure efficiency:

- **OTR of 8.42** is the lowest in our universe — roughly 1 trade for every 8 orders submitted. This reflects a market dominated by directional order flow and sophisticated participants who execute rather than quote.
- **fill_rate of 11.9%** is the highest in our universe — nearly 1 in 8 orders actually trades, versus 1 in 50 on FDAX or 1 in 300 on NKD. This is the signature of a genuinely liquid market.
- **885K trades** in a single day, with ~20M total events — the raw throughput dwarfs all other products in our universe.
- **cancel_rate of 99.9%** confirms that the vast majority of order activity is quote maintenance by HFT market makers, not genuine trading intent.
- **n_modify of 2M** — ES has a significant MODIFY action count, reflecting CME's support for order modification. HKEX products show zero MODIFYs (the exchange implements modify as Delete+Add).
- **Traded price range: 6540–6807 pts** (~4% intraday range) — a normal volatility day in Q4 2025.


### Europe — Eurex

In [11]:
df_eu = dr.get_cross_market_summary(["FDAX", "FESX", "FSMI"], "2025-05-14")
df_eu.insert(0, "zone", "EU")
df_eu

,zone,product,contract,date,exchange,currency,n_rows,n_add,n_cancel,n_modify,n_trade,otr,cancel_rate,fill_rate,price_min_pts,price_max_pts,tick_size_pts,point_value
0,EU,FDAX,FDAXM25,2025-05-14,EUREX,EUR,2324424,1134105,1134849,4460,25512,44.45,1.0007,0.0225,23500.0,23786.0,0.5,25.0
1,EU,FESX,FESXM25,2025-05-14,EUREX,EUR,2735342,1251162,1267873,34314,91063,13.74,1.0134,0.0728,5364.0,5412.0,1.0,10.0
2,EU,FSMI,FSMIM25,2025-05-14,EUREX,CHF,519833,241512,242277,8106,13968,17.29,1.0032,0.0578,12071.0,12165.0,1.0,10.0


**FDAX, FESX, FSMI — 2025-05-14**

The three Eurex products reveal a striking disparity within the same exchange:

**FESX vs FDAX — the OTR anomaly:**  
The Euro Stoxx 50 (OTR 13.74) trades three times more efficiently than the DAX (OTR 44.45), despite both being EUR-denominated equity index futures on Eurex. This is a genuine microstructure difference:
- FESX is a broader European index with much higher institutional hedging demand — pension funds, asset managers, and index replicators use it as their primary European equity hedge. This drives higher directional order flow relative to pure market-making noise.
- FDAX is more dominated by HFT market makers who submit and cancel large numbers of orders to maintain tight spreads. The higher OTR reflects this quote-maintenance behaviour.
- FESX `cancel_rate > 1` (1.013) is explained by overnight GTC orders cancelled during our window without a matching ADD in the daily file — a structural artefact of our daily data partitioning in LOOSE mode.

**FSMI — the illiquid outlier:**  
The SMI Future has an intermediate OTR of 17.29 and only ~14K trades/day — roughly 3.5x fewer than FESX. This reflects the smaller size of the Swiss equity market and lower international hedging demand. Its presence in our universe is valuable precisely because it anchors the lower end of the European liquidity spectrum.

**n_modify on Eurex:**  
FESX shows 34K modify events versus only 4.5K on FDAX — a factor of 7x. This suggests FESX participants adjust existing order sizes more actively rather than cancelling and resubmitting, consistent with a more directional order flow profile.


### Asia — CME Globex + HKEX

In [12]:
df_asia = dr.get_cross_market_summary(["NIY", "NKD", "HSI", "MHI", "HHI", "MCH"], "2026-02-02")
df_asia.insert(0, "zone", "Asia")
df_asia

,zone,product,contract,date,exchange,currency,n_rows,n_add,n_cancel,n_modify,n_trade,otr,cancel_rate,fill_rate,price_min_pts,price_max_pts,tick_size_pts,point_value
0,Asia,NIY,NIYH26,2026-02-02,CME,JPY,1760388,796541,795361,131343,15433,51.61,0.9985,0.0194,52555.0,54275.0,5.0,500.0
1,Asia,NKD,NKDH26,2026-02-02,CME,USD,524175,244683,244187,26576,3995,61.25,0.9980,0.0163,52635.0,54365.0,5.0,5.0
2,Asia,HSI,HSIG26,2026-02-02,HKEX,HKD,8792881,4393936,4269769,0,129172,34.02,0.9717,0.0294,26543.0,27247.0,1.0,50.0
3,Asia,MHI,MHIG26,2026-02-02,HKEX,HKD,6182611,3089654,2990945,0,102008,30.29,0.9681,0.0330,26544.0,27243.0,1.0,10.0
4,Asia,HHI,HHIG26,2026-02-02,HKEX,HKD,4186553,2086613,2005977,0,93959,22.21,0.9614,0.0450,9003.0,9281.0,1.0,50.0
5,Asia,MCH,MCHG26,2026-02-02,HKEX,HKD,1097642,549171,538977,0,9490,57.87,0.9814,0.0173,9004.0,9279.0,1.0,10.0


**NIY / NKD — Nikkei 225, two currencies (CME Globex) — 2026-02-02**

NIY (JPY-denominated) and NKD (USD-denominated) track the same underlying index but attract fundamentally different participant bases:

- **NIY dominates in volume:** 15.4K trades vs 4.0K for NKD (~3.9x more), with 3.4x more events overall. Japanese domestic institutions and Tokyo-based HFT firms primarily use NIY. The JPY denomination provides natural currency alignment for these participants.
- **NKD has a higher OTR (61.25 vs 51.61):** the USD-denominated contract attracts more international market makers who submit many more quotes relative to actual trades — consistent with a structurally thinner book.
- **NIY has significantly more MODIFY events (131K vs 27K):** NIY participants adjust existing orders rather than cancelling and resubmitting, suggesting a different order management strategy — possibly reflecting longer-horizon participants who adjust limit prices as the market moves.
- **Price ranges are nearly identical** (NIY: 52555–54275, NKD: 52635–54365) — same underlying index. The small difference is the JPY/USD FX translation effect.

**HSI / MHI — Hang Seng Index, standard vs mini (HKEX) — 2026-02-02**

HSI and MHI track the identical underlying with a 5:1 contract size ratio (HKD 50/pt vs HKD 10/pt):

- **HSI leads in absolute trade count** (129K vs 102K) despite being the larger contract, consistent with institutional dominance.
- **MHI has a slightly lower OTR (30.29 vs 34.02) and higher fill_rate (3.3% vs 2.9%):** the mini contract is marginally more efficient in our metrics, reflecting a higher proportion of retail and small-account traders who submit orders with genuine execution intent.
- **Price ranges are identical** (HSI: 26543–27247, MHI: 26544–27243) — same underlying. Sub-point differences are rounding in fixed-point to float conversion.
- **n_modify = 0 for all HKEX products** — HKEX's OMD-D protocol does not support order modification. Participants must cancel and resubmit (Delete+Add), which inflates both n_cancel and n_add counts.

**HHI / MCH — H-shares China Enterprises Index, standard vs mini (HKEX) — 2026-02-02**

HHI and MCH track H-shares (Chinese companies listed in Hong Kong), providing pure China exposure:

- **HHI has the lowest OTR in the HKEX universe (22.21)** — lower even than HSI. H-shares attract more directional China macro traders, resulting in a higher proportion of genuine execution relative to market-making noise.
- **MCH has the highest OTR (57.87)** — the mini H-shares contract shows more noise than the standard. This may reflect a different participant mix: more local retail market makers using the mini as a hedging vehicle. This inversion (mini noisier than standard) contrasts with the HSI/MHI pair and warrants further investigation in Phase 3.
- **Lower absolute trade counts** (HHI: 94K, MCH: 9.5K) vs HSI/MHI — the H-shares index is less internationally recognised and attracts a narrower participant base.


### Combined Summary — All Zones

In [13]:
df_all = pd.concat([df_us, df_eu, df_asia], ignore_index=True)

# Display key microstructure metrics sorted by OTR within each zone
cols = ["zone", "product", "exchange", "currency", "n_trade", "otr", "cancel_rate", "fill_rate", "price_min_pts", "price_max_pts"]
df_all[cols].sort_values(["zone", "otr"])

,zone,product,exchange,currency,n_trade,otr,cancel_rate,fill_rate,price_min_pts,price_max_pts
8,Asia,HHI,HKEX,HKD,93959,22.21,0.9614,0.0450,9003.00,9281.0
7,Asia,MHI,HKEX,HKD,102008,30.29,0.9681,0.0330,26544.00,27243.0
6,Asia,HSI,HKEX,HKD,129172,34.02,0.9717,0.0294,26543.00,27247.0
4,Asia,NIY,CME,JPY,15433,51.61,0.9985,0.0194,52555.00,54275.0
9,Asia,MCH,HKEX,HKD,9490,57.87,0.9814,0.0173,9004.00,9279.0
5,Asia,NKD,CME,USD,3995,61.25,0.9980,0.0163,52635.00,54365.0
2,EU,FESX,EUREX,EUR,91063,13.74,1.0134,0.0728,5364.00,5412.0
3,EU,FSMI,EUREX,CHF,13968,17.29,1.0032,0.0578,12071.00,12165.0
1,EU,FDAX,EUREX,EUR,25512,44.45,1.0007,0.0225,23500.00,23786.0
0,US,ES,CME,USD,885379,8.42,0.9992,0.1188,6540.25,6806.5


**Cross-market interpretation**

Sorting by OTR within each zone reveals a clear liquidity hierarchy:

| OTR range | Interpretation |
|-----------|---------------|
| < 15 | High directional flow, institutional-grade liquidity (ES, FESX) |
| 15–35 | Mixed — market making + directional (FSMI, HHI, MHI, HSI) |
| 35–65 | HFT-dominated quoting, lower execution probability (FDAX, NIY, MCH, NKD) |

**Key cross-market observations:**

1. **ES is a structural outlier** — its OTR of 8.42 is 1.6x lower than the next most efficient product (FESX at 13.74). The combination of deep liquidity, near-24h trading, and the world's most active options hedging flow creates a fundamentally different microstructure.

2. **The FESX/FDAX divergence within Eurex** (OTR 13.74 vs 44.45) is the most striking intra-exchange anomaly in our universe. Both products trade on the same matching engine, same connectivity, same participants — yet their order flow composition differs by a factor of 3x. This is a research-grade microstructure question: does FESX's higher fill rate reflect structural differences in participant types, tick size relative to volatility, or index composition effects?

3. **HKEX products cluster in the 22–58 OTR range** with no MODIFY events and cancel rates around 96–98%. The standard/mini pairs (HSI/MHI, HHI/MCH) show the same underlying price but meaningfully different OTR, pointing to participant fragmentation effects.

4. **cancel_rate is not a clean metric on Eurex** (values slightly above 1.0) due to overnight GTC orders. OTR is the more robust cross-exchange comparison metric.

> **Note on dates:** The three zones use different reference dates. Cross-zone comparisons should account for potential regime differences between periods. A common date analysis will be possible once additional data is acquired.

**Next steps:**
- OFI (Order Flow Imbalance) computation per product
- VPIN toxicity measure
- Rolling OTR / cancel_rate time series to detect regime changes (Phase 4)
